# Composition baseline — single dataset walkthrough

How to run the **global cell-type composition** baseline for one dataset, under both
validation schemes:

1. **Cross-validation (CV)** — patient-level K-fold *within* a cohort.
2. **Cohort-split generalization** — train on one cohort, test on another.

The benchmark is organised so the *only* things that change between baseline families
(global / local-MIL / GNN) are the **Featurizer** and the **Model**; the data splits,
target building, metric scoring and cross-cohort alignment are shared in
`benchmark.validation.suite`. Here we use:

| piece | class | module |
|---|---|---|
| Featurizer | `CompositionFeaturizer` | `benchmark.features.global_feats` |
| Model | `LinearClassifier` / `LinearCox` | `benchmark.models.linear` |
| Runner | `run_cross_validation`, `run_generalization` | `benchmark.validation.suite` |

We use **HNC-Wu2022** because it has both a CV cohort (UPMC) and a declared
generalization test (UPMC → DFCI).

## 0. Setup — put the `benchmark` package on the path

In [3]:
import sys
from pathlib import Path

# data_root = Path(".").resolve().parent.parent
data_root = '/autofs/bal14/zqwu/CellularTables/TME_benchmark_data'
code_root = Path(".").resolve().parent
sys.path.insert(0, str(code_root))

print(f"Added {code_root} to sys.path for imports")
print(f"Data root: {data_root}")

Added /autofs/nas8/tywang/tjzou/TME_modeling_benchmark to sys.path for imports
Data root: /autofs/bal14/zqwu/CellularTables/TME_benchmark_data


## 1. Load the dataset and inspect its validation config

In [5]:
from benchmark.utils.registry import load_dataset

ds = load_dataset("hnc_wu2022", data_root=data_root)
print("Dataset :", ds)
print("Tasks   :", ds.task_ids)

vcfg = ds.validation_config
print("\nCV cohort filter        :", vcfg.get("cv_filter"))
print("n_folds                 :", vcfg.get("n_folds"))
print("cohort column           :", vcfg.get("cohort_col"))
print("generalization_tests    :")
for gt in vcfg.get("generalization_tests", []):
    print("   ", gt)

Dataset : TMEDataset(name='HNC-Wu2022', root=/autofs/bal14/zqwu/CellularTables/TME_benchmark_data/HNC_Wu2022/processed)
Tasks   : ['primary_outcome', 'hpv_status', 'OS']

CV cohort filter        : dataset == 'UPMC_HNC'
n_folds                 : 5
cohort column           : dataset
generalization_tests    :
    {'name': 'UPMC_to_DFCI', 'train': ['UPMC_HNC'], 'test': ['DFCI_HNC'], 'tasks': ['primary_outcome', 'OS'], 'cell_type_col': 'cell_type_uniform'}


## 2. The two building blocks: Featurizer + Model

The **Featurizer** maps each region to a representation. `CompositionFeaturizer`
produces one **tabular** vector per region (cell-type fractions). `fit` learns the
cell-type vocabulary from the *training* regions only (no leakage); `transform`
builds the region × cell-type matrix.

In [6]:
from benchmark.features.basic_feats import CompositionFeaturizer, MeanExpressionFeaturizer
from benchmark.features.density_feats import CellTypeDensityFeaturizer

task_cfg = ds.get_task_config("primary_outcome")
meta = ds.get_task_metadata("primary_outcome")

upmc = meta.query("dataset == 'UPMC_HNC'")["region_id"].tolist()

meta.head(3)

,region_id,patient_id,dataset,primary_outcome,hpv_status,OS_event,OS_months,pfs_status,pfs_months,ptr_label
0,UPMC_c001_v001_r001_reg001,67653.0,UPMC_HNC,0.0,0.0,1.0,32.89,NaN,NaN,NaN
1,UPMC_c001_v001_r001_reg002,67651.0,UPMC_HNC,0.0,0.0,1.0,27.30,NaN,NaN,NaN
2,UPMC_c001_v001_r001_reg004,67629.0,UPMC_HNC,0.0,0.0,1.0,2.00,NaN,NaN,NaN


In [8]:
print("Demonstrating different feature extractors:")

feat = CompositionFeaturizer().fit(ds.load_regions(upmc))
print(f"===\nComposition of {len(feat.cell_types_)} cell types:")
X1 = feat.transform(ds.load_regions(upmc[:3]))
display(X1.head(3))

feat = MeanExpressionFeaturizer().fit(ds.load_regions(upmc))
print(f"===\nMean expression of {len(feat.markers_)} biomarkers:")
X1 = feat.transform(ds.load_regions(upmc[:3]))
display(X1.head(3))

feat = CellTypeDensityFeaturizer().fit(ds.load_regions(upmc))
print(f"===\nDensity of {len(feat.cell_types_)} cell types in tissue & tumor (counts per mm^2):")
X1 = feat.transform(ds.load_regions(upmc[:3]))
display(X1.head(3))

Demonstrating different feature extractors:
===
Composition of 17 cell types:


,APC,B cell,CD4 T cell,CD8 T cell,Granulocyte,Lymph vessel,Macrophage,Naive immune cell,Stromal / Fibroblast,Tumor,Tumor (CD15+),Tumor (CD20+),Tumor (CD21+),Tumor (Ki67+),Tumor (Podo+),Unassigned,Vessel
region_id,,,,,,,,,,,,,,,,,
UPMC_c001_v001_r001_reg001,0.019037,0.005734,0.033486,0.029587,0.058945,0.009862,0.114679,0.014220,0.176606,0.113991,0.019954,0.068578,0.126835,0.059862,0.105963,0.013532,0.029128
UPMC_c001_v001_r001_reg002,0.008333,0.004023,0.044971,0.019828,0.012356,0.000718,0.063506,0.019828,0.141523,0.039511,0.082759,0.061782,0.135632,0.167098,0.134626,0.007759,0.055747
UPMC_c001_v001_r001_reg004,0.030024,0.014304,0.052259,0.088798,0.014870,0.000283,0.084124,0.026908,0.032007,0.229571,0.033423,0.029458,0.070528,0.147571,0.094038,0.005665,0.046169


===
Mean expression of 41 biomarkers:


,CD117,CD11b,CD11c,CD134,CD14,CD15,CD152,CD16,CD163,CD20,...,ICOS,Ki67,PD1,PDL1,PanCK,Podoplanin,TMEM16A,Vimentin,aSMA,p16
region_id,,,,,,,,,,,,,,,,,,,,,
UPMC_c001_v001_r001_reg001,0.291031,0.404538,0.297905,0.281916,0.499058,0.515226,0.252735,0.273734,0.632742,0.354262,...,0.236689,0.331689,0.269268,0.319820,1.044944,0.485416,0.477133,0.505659,0.412763,0.244838
UPMC_c001_v001_r001_reg002,0.259444,0.336526,0.229852,0.237063,0.345829,0.463145,0.228826,0.232968,0.789235,0.308982,...,0.243545,0.544476,0.233342,0.266993,1.043755,0.720537,0.649360,1.066624,0.319403,0.269902
UPMC_c001_v001_r001_reg004,0.243988,0.334101,0.237119,0.235016,0.337123,0.397116,0.225167,0.243083,0.366188,0.303361,...,0.238701,0.510163,0.234617,0.233866,0.311855,0.546677,0.372414,0.846442,0.304729,0.230519


===
Density of 17 cell types in tissue & tumor (counts per mm^2):


,tissue_density::APC,tissue_density::B cell,tissue_density::CD4 T cell,tissue_density::CD8 T cell,tissue_density::Granulocyte,tissue_density::Lymph vessel,tissue_density::Macrophage,tissue_density::Naive immune cell,tissue_density::Stromal / Fibroblast,tissue_density::Tumor,...,tumor_density::Naive immune cell,tumor_density::Stromal / Fibroblast,tumor_density::Tumor,tumor_density::Tumor (CD15+),tumor_density::Tumor (CD20+),tumor_density::Tumor (CD21+),tumor_density::Tumor (Ki67+),tumor_density::Tumor (Podo+),tumor_density::Unassigned,tumor_density::Vessel
region_id,,,,,,,,,,,,,,,,,,,,,
UPMC_c001_v001_r001_reg001,0.108808,0.032773,0.191397,0.167800,0.336911,0.056370,0.655469,0.081278,1.005490,0.650226,...,0.068539,0.918422,0.744790,0.127939,0.453880,0.842267,0.389910,0.693005,0.062447,0.147739
UPMC_c001_v001_r001_reg002,0.075601,0.036497,0.407984,0.179878,0.112098,0.006517,0.576131,0.178575,1.282609,0.358453,...,0.082248,0.493485,0.524089,1.084520,0.822476,1.803708,2.216859,1.788407,0.076509,0.233354
UPMC_c001_v001_r001_reg004,0.288065,0.137239,0.501396,0.851966,0.141315,0.002718,0.807126,0.258172,0.307088,2.202611,...,0.103471,0.101949,2.466556,0.357582,0.316498,0.756248,1.582491,1.002752,0.042606,0.319542


The **Model** consumes a feature dataframe + a region-indexed target and predicts
per region.

In [9]:
from benchmark.validation.splits import safe_patient_kfold, stratify_column
cv_meta = meta.query(vcfg["cv_filter"]).reset_index(drop=True)
folds = safe_patient_kfold(cv_meta, n_folds=5, patient_col="patient_id",
                           stratify_col=stratify_column(task_cfg), seed=0)

# Use one fold as an example
train_ids, val_ids = folds[0]

In [10]:
from benchmark.models.forest import RandomForestModel
from benchmark.validation.metrics import score_predictions

task = "primary_outcome"
task_cfg = ds.get_task_config(task)

feat = CellTypeDensityFeaturizer().fit(ds.load_regions(train_ids))
X_tr, X_va = feat.transform(ds.load_regions(train_ids)), feat.transform(ds.load_regions(val_ids))

y_tr = ds.build_target(train_ids, task)
y_va = ds.build_target(val_ids, task)

model = RandomForestModel(seed=0).fit(X_tr, y_tr)
proba_tr = model.predict(X_tr)
proba_va = model.predict(X_va)

print(f"Task: {task}")
print(f"trained on {len(X_tr)} regions")
print(score_predictions(task_cfg, y_tr, proba_tr, model.classes_))

print(f"validated on {len(X_va)} regions")
print(score_predictions(task_cfg, y_va, proba_va, model.classes_))

Task: primary_outcome
trained on 243 regions
{'auc_roc': 1.0, 'avg_precision': 1.0, 'balanced_acc': 1.0}
validated on 65 regions
{'auc_roc': 0.6765, 'avg_precision': 0.7695064141939862, 'balanced_acc': 0.6425000000000001}


In [11]:
from benchmark.models.linear import LinearCox

task = "OS"
task_cfg = ds.get_task_config(task)

feat = CellTypeDensityFeaturizer().fit(ds.load_regions(train_ids))
X_tr, X_va = feat.transform(ds.load_regions(train_ids)), feat.transform(ds.load_regions(val_ids))

y_tr = ds.build_target(train_ids, task)
y_va = ds.build_target(val_ids, task)

model = LinearCox(seed=0).fit(X_tr, y_tr)
risk_tr = model.predict(X_tr)
risk_va = model.predict(X_va)

print(f"Task: {task}")
print(f"trained on {len(X_tr)} regions")
print(score_predictions(task_cfg, y_tr, risk_tr, model.classes_))

print(f"validated on {len(X_va)} regions")
print(score_predictions(task_cfg, y_va, risk_va, model.classes_))

Task: OS
trained on 243 regions
{'c_index': 0.7989377314976588}
validated on 65 regions
{'c_index': 0.6068237205523964}


## 3. Cross-validation over all tasks (within UPMC)

In [12]:
import pandas as pd
from benchmark.validation.crossval import cross_validate
from benchmark.validation.metrics import PRIMARY_METRIC, summarize_folds
from benchmark.models.linear import LinearCox
from benchmark.models.forest import RandomForestModel

featurizer = CellTypeDensityFeaturizer().fit(ds.load_regions(ds.all_region_ids))

def model_factory(task_cfg, seed):
    return LinearCox(seed=seed) if task_cfg["type"] == "survival" else RandomForestModel(seed=seed)

for task_id in ds.task_ids:
    task_cfg = ds.get_task_config(task_id)
    print(f"\n===\nTask: {task_id} ({task_cfg['type']})")

    fold_metrics = cross_validate(
        ds, task_id, featurizer, model_factory, seeds=[0, 1, 2],
    )
    fm = pd.DataFrame(fold_metrics)
    print("folds:", len(fm), "(3 seeds x 5 folds)")
    
    mean, sd = summarize_folds(fold_metrics, PRIMARY_METRIC[task_cfg["type"]])
    print(f"primary_outcome CV {PRIMARY_METRIC[task_cfg["type"]]}: {mean:.4f} +/- {sd:.4f}")


===
Task: primary_outcome (binary_classification)
folds: 15 (3 seeds x 5 folds)
primary_outcome CV auc_roc: 0.7258 +/- 0.0767

===
Task: hpv_status (binary_classification)
folds: 15 (3 seeds x 5 folds)
primary_outcome CV auc_roc: 0.8217 +/- 0.1000

===
Task: OS (survival)
folds: 15 (3 seeds x 5 folds)
primary_outcome CV c_index: 0.6863 +/- 0.0932


## 4. Cohort-split generalization (UPMC → DFCI)

In [13]:
meta = ds.get_task_metadata(task)

gentest_cfg = vcfg["generalization_tests"][0]
cohort_col = vcfg["cohort_col"]

# split by COHORT (not patient fold): train cohort(s) -> held-out test cohort(s)
train_ids = meta[meta[cohort_col].isin(gentest_cfg["train"])]["region_id"].tolist()
test_ids  = meta[meta[cohort_col].isin(gentest_cfg["test"])]["region_id"].tolist()
ct_col = gentest_cfg.get("cell_type_col", ds.config['cell_type_col'])
print(f"train cohorts: {gentest_cfg['train']}, n={len(train_ids)}")
print(f"test cohorts: {gentest_cfg['test']}, n={len(test_ids)}")
print(f"Using cell_type_col: {ct_col}")

train cohorts: ['UPMC_HNC'], n=308
test cohorts: ['DFCI_HNC'], n=58
Using cell_type_col: cell_type_uniform


In [ ]:
from benchmark.models.forest import RandomForestModel
from benchmark.validation.metrics import score_predictions

task = "primary_outcome"
task_cfg = ds.get_task_config(task)

### Define featurizer, note that we fit the featurizer on the combined train+test set to ensure consistent feature columns across cohorts.
### Also note that `ct_col` is pointing to "cell_type_uniform", which is the harmonized cell type column across cohorts.
feat = CompositionFeaturizer(cell_type_col=ct_col).fit(ds.load_regions(train_ids + test_ids))
# feat = CellTypeDensityFeaturizer(cell_type_col=ct_col).fit(ds.load_regions(train_ids + test_ids))
# feat = MeanExpressionFeaturizer().fit(ds.load_regions(train_ids + test_ids))

X_tr, X_te = feat.transform(ds.load_regions(train_ids)), feat.transform(ds.load_regions(test_ids))

y_tr = ds.build_target(train_ids, task)
y_te = ds.build_target(test_ids, task)

model = RandomForestModel(seed=0).fit(X_tr, y_tr)
proba_tr = model.predict(X_tr)
proba_te = model.predict(X_te)

print(f"Task: {task}  ({gentest_cfg['name']}, features = {ct_col})")
print(f"trained on {len(X_tr)} {gentest_cfg['train']} regions")
print(score_predictions(task_cfg, y_tr, proba_tr, model.classes_))

print(f"tested on  {len(X_te)} {gentest_cfg['test']} regions")
print(score_predictions(task_cfg, y_te, proba_te, model.classes_))

Task: primary_outcome  (UPMC_to_DFCI, features = cell_type_uniform)
trained on 308 ['UPMC_HNC'] regions
{'auc_roc': 1.0, 'avg_precision': 1.0, 'balanced_acc': 1.0}
tested on  58 ['DFCI_HNC'] regions
{'auc_roc': 0.7448484848484849, 'avg_precision': 0.598381742918528, 'balanced_acc': 0.5909090909090909}


In [28]:
from benchmark.models.linear import LinearCox

task = "OS"
task_cfg = ds.get_task_config(task)

feat = CompositionFeaturizer(cell_type_col=ct_col).fit(ds.load_regions(train_ids + test_ids))
# feat = CellTypeDensityFeaturizer(cell_type_col=ct_col).fit(ds.load_regions(train_ids + test_ids))
# feat = MeanExpressionFeaturizer().fit(ds.load_regions(train_ids + test_ids))

X_tr, X_te = feat.transform(ds.load_regions(train_ids)), feat.transform(ds.load_regions(test_ids))

y_tr = ds.build_target(train_ids, task)
y_te = ds.build_target(test_ids, task)

model = LinearCox(seed=0).fit(X_tr, y_tr)
proba_tr = model.predict(X_tr)
proba_te = model.predict(X_te)

print(f"Task: {task}  ({gentest_cfg['name']}, features = {ct_col})")
print(f"trained on {len(X_tr)} {gentest_cfg['train']} regions")
print(score_predictions(task_cfg, y_tr, proba_tr, model.classes_))

print(f"tested on  {len(X_te)} {gentest_cfg['test']} regions")
print(score_predictions(task_cfg, y_te, proba_te, model.classes_))

Task: OS  (UPMC_to_DFCI, features = cell_type_uniform)
trained on 308 ['UPMC_HNC'] regions
{'c_index': 0.7241180823945399}
tested on  58 ['DFCI_HNC'] regions
{'c_index': 0.4798807749627422}


## 5. Wrapper for the test

`cohort_split_test` is the generalization counterpart to `cross_validate`: it trains on
the test's train cohort(s) and evaluates on the held-out test cohort(s) — the manual
cell in part 4, in one call — repeated over **seeds** (the split is fixed; seeds only
re-seed the model). Same factories; just bake the test's `cell_type_col` into the
featurizer so the feature space transfers across cohorts.

In [29]:
import pandas as pd
from benchmark.validation.crossval import cohort_split_test
from benchmark.features.basic_feats import CompositionFeaturizer
from benchmark.validation.metrics import PRIMARY_METRIC, summarize_folds
from benchmark.models.linear import LinearCox
from benchmark.models.forest import RandomForestModel

gentest_cfg = vcfg["generalization_tests"][0]

all_regions = ds.all_region_ids

featurizer = CompositionFeaturizer(cell_type_col=gentest_cfg.get("cell_type_col", "cell_type"))
featurizer.fit(ds.load_regions(all_regions))

def model_factory(task_cfg, seed):
    return LinearCox(seed=seed) if task_cfg["type"] == "survival" else RandomForestModel(seed=seed)

for task_id in gentest_cfg['tasks']:
    task_cfg = ds.get_task_config(task_id)
    print(f"\n===\nTask: {task_id} ({task_cfg['type']})")

    results = cohort_split_test(
        ds, task_id, gentest_cfg, featurizer, model_factory, seeds=[0, 1, 2],
    )

    mean, sd = summarize_folds(results, PRIMARY_METRIC[task_cfg["type"]])
    print(f"Cohort-split test for {task_id}: {PRIMARY_METRIC[task_cfg["type"]]}: {mean:.4f} +/- {sd:.4f}")


===
Task: primary_outcome (binary_classification)
Cohort-split test for primary_outcome: auc_roc: 0.7400 +/- 0.0228

===
Task: OS (survival)
Cohort-split test for OS: c_index: 0.4799 +/- 0.0000
